# Feedforward Networks I: The Forward Pass and Activation Functions
## Units, nonlinearities, and how information flows through a network

**MAT 4953/6973 — Mathematical Foundations of AI** (Spring 2026, UTSA)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/fwdpass_units.ipynb)

---

> **Prerequisites:** Vectors and matrices (matrix–vector products, norms), basic calculus (derivatives, chain rule), Python/NumPy. Familiarity with [`mnist_ff.ipynb`](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/mnist_ff.ipynb) is assumed.

In [`mnist_ff.ipynb`](https://colab.research.google.com/github/eduenez/MathAIspring2026UTSA/blob/main/code/mnist_ff.ipynb) we built a feedforward network and trained it on MNIST using Keras.
We wrote `layers.Dense(64, activation="relu")` without stopping to ask:

- What exactly *is* a unit (neuron)?
- Why ReLU and not something else — sigmoid, tanh, GeLU?
- How do the weight matrices $W^{(l)}$ and biases $\mathbf{b}^{(l)}$ combine to turn input $\mathbf{x}$ into output $\hat{\mathbf{y}}$?

This notebook answers those questions. The **forward pass** — computing $\hat{\mathbf{y}}$ from $\mathbf{x}$ — is the foundation for everything else: loss evaluation, gradient computation, and training.

**Outline**
1. [Feedforward architecture — a quick recap](#part1)
2. [The forward pass, layer by layer](#part2)
3. [Activation functions — a tour of units](#part3)
4. [Tracing a complete forward pass](#part4)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from ipywidgets import interact, FloatSlider, Dropdown
from scipy.special import ndtr          # standard normal CDF, used for GeLU
from scipy.stats import norm as _norm   # for GeLU derivative

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13, 'figure.dpi': 100})
rng = np.random.default_rng(42)
print("Libraries loaded.")

<a id="part1"></a>
# Part 1: Feedforward Architecture — A Quick Recap

## 1.1 Layers, weights, and biases

A **feedforward network** (or **multi-layer perceptron**, MLP) is an ordered sequence of layers:

$$\underbrace{\mathbf{x}}_{\text{input}} \;\longrightarrow\; \underbrace{\text{layer 1}}_{\text{hidden}} \;\longrightarrow\; \cdots \;\longrightarrow\; \underbrace{\text{layer } (L-1)}_{\text{hidden}} \;\longrightarrow\; \underbrace{\hat{\mathbf{y}}}_{\text{output}}$$

Each layer $l \in \{1, \ldots, L\}$ has **width** $n_l$ (number of units). Layer $l-1$ connects to layer $l$ through:
- a **weight matrix** $W^{(l)} \in \mathbb{R}^{n_l \times n_{l-1}}$, and
- a **bias vector** $\mathbf{b}^{(l)} \in \mathbb{R}^{n_l}$.

In the MNIST model we used the architecture $(n_0, n_1, n_2, n_3) = (784, 64, 64, 10)$.
The figure below shows a scaled-down version with the same structure.

In [ ]:
def draw_network(layer_sizes, ax, xs=2.4, ys=1.05, r=0.27, max_n=8):
    # Draw a feedforward network diagram on ax.
    L = len(layer_sizes)
    node_pos = {}
    clr  = ['#d5e8d4'] + ['#dae8fc'] * (L - 2) + ['#fff2cc']
    eclr = ['#82b366'] + ['#6c8ebf'] * (L - 2) + ['#d6b656']
    for l, n in enumerate(layer_sizes):
        shown = min(n, max_n)
        ys_arr = np.linspace(-(shown - 1) / 2 * ys, (shown - 1) / 2 * ys, shown)
        x = l * xs
        for j, y in enumerate(ys_arr):
            node_pos[(l, j)] = (x, y)
            ax.add_patch(plt.Circle((x, y), r, color=clr[l], ec=eclr[l], lw=1.5, zorder=3))
        if n > max_n:
            ax.text(x, -(shown) / 2 * ys - 0.15, r'$\vdots$',
                    ha='center', va='top', fontsize=13)
    for l in range(L - 1):
        for j in range(min(layer_sizes[l], max_n)):
            for k in range(min(layer_sizes[l + 1], max_n)):
                x0, y0 = node_pos[(l, j)]
                x1, y1 = node_pos[(l + 1, k)]
                ax.plot([x0 + r, x1 - r], [y0, y1], 'k-', alpha=0.10, lw=0.7, zorder=1)
    return node_pos

fig, ax = plt.subplots(figsize=(10, 5))
pos = draw_network([4, 5, 5, 3], ax)
ax.set_xlim(-0.7, 3 * 2.4 + 0.7)
ax.set_ylim(-3.5, 4.2)
ax.set_aspect('equal')
ax.axis('off')

layer_labels = [r'Input$\;\mathbf{x}$'+'\n'+r'$n_0 = 4$',
                r'Hidden 1'+'\n'+r'$n_1 = 5$',
                r'Hidden 2'+'\n'+r'$n_2 = 5$',
                r'Output$\;\hat{\mathbf{y}}$'+'\n'+r'$n_3 = 3$']
for l, lbl in enumerate(layer_labels):
    ax.text(l * 2.4, 3.6, lbl, ha='center', fontsize=11)

patches = [mpatches.Patch(fc='#d5e8d4', ec='#82b366', label='Input layer'),
           mpatches.Patch(fc='#dae8fc', ec='#6c8ebf', label='Hidden layers'),
           mpatches.Patch(fc='#fff2cc', ec='#d6b656', label='Output layer')]
ax.legend(handles=patches, loc='lower right', fontsize=10)
ax.set_title('Feedforward network with 2 hidden layers  (scaled-down MNIST architecture)',
             fontsize=12)
plt.tight_layout()
plt.show()

## 1.2 What is a single unit?

Each **unit** $k$ in layer $l$ computes its output in two steps:

**Step 1 — Pre-activation** (linear):
$$z_k^{(l)} = \sum_{j=1}^{n_{l-1}} W^{(l)}_{kj}\, a_j^{(l-1)} + b_k^{(l)}.$$

**Step 2 — Activation** (nonlinear):
$$a_k^{(l)} = \sigma\!\left(z_k^{(l)}\right).$$

Written for all $n_l$ units in layer $l$ simultaneously (using matrix–vector notation):

$$\boxed{\mathbf{z}^{(l)} = W^{(l)}\,\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}, \qquad \mathbf{a}^{(l)} = \sigma\!\left(\mathbf{z}^{(l)}\right).}$$

Here $\sigma$ is applied **element-wise**: $\sigma(\mathbf{z})_k = \sigma(z_k)$.
The choice of $\sigma$ defines the *type* of unit; [Part 3](#part3) surveys the most important choices.

<a id="part2"></a>
# Part 2: The Forward Pass

## 2.1 Layer by layer

Setting $\mathbf{a}^{(0)} = \mathbf{x}$, the network applies:
$$\mathbf{z}^{(l)} = W^{(l)}\,\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}, \qquad \mathbf{a}^{(l)} = \sigma^{(l)}\!\left(\mathbf{z}^{(l)}\right), \qquad l = 1, \ldots, L.$$

The output is $\hat{\mathbf{y}} = \mathbf{a}^{(L)}$.
The full computation is the composition $F^{(L)} \circ \cdots \circ F^{(1)}$ where each *layer map* is:
$$F^{(l)}(\mathbf{a}) = \sigma^{(l)}\!\left(W^{(l)}\mathbf{a} + \mathbf{b}^{(l)}\right).$$
This is exactly what `model.predict(x)` does in `mnist_ff.ipynb`.

## 2.2 Why nonlinearity is indispensable

**Claim:** if every $\sigma^{(l)}$ is the identity ($\mathbf{a}^{(l)} = \mathbf{z}^{(l)}$), the entire $L$-layer network collapses to a *single* affine map.

**Proof:** with $\sigma = \mathrm{Id}$ we can substitute layer by layer:
$$\hat{\mathbf{y}} = W^{(L)}\!\left(\cdots\!\left(W^{(2)}\!\left(W^{(1)}\mathbf{x} + \mathbf{b}^{(1)}\right) + \mathbf{b}^{(2)}\right)\!\cdots\right) + \mathbf{b}^{(L)} = \underbrace{W^{(L)}\cdots W^{(1)}}_{\widetilde{W}}\,\mathbf{x} + \widetilde{\mathbf{b}}.$$

Depth without nonlinearity adds no representational power — the network can only learn linear functions.
The code below lets you verify this empirically.

In [ ]:
# Verify: stacking linear layers = one linear layer
rng2 = np.random.default_rng(7)
n0, n1, n2 = 3, 5, 2

W1 = rng2.standard_normal((n1, n0))
b1 = rng2.standard_normal(n1)
W2 = rng2.standard_normal((n2, n1))
b2 = rng2.standard_normal(n2)
x  = rng2.standard_normal(n0)

# Two-layer forward pass, identity activation
a1 = W1 @ x + b1        # z1 = a1 (no nonlinearity)
y_deep = W2 @ a1 + b2   # z2 = y_deep

# Equivalent single layer
W_tilde = W2 @ W1
b_tilde = W2 @ b1 + b2
y_flat  = W_tilde @ x + b_tilde

print(f"Two-layer output:  {y_deep}")
print(f"Single-layer output: {y_flat}")
print(f"Are they equal?    {np.allclose(y_deep, y_flat)}")

> **Exercise 2.1.** *(Linear networks collapse)*
>
> **(a)** The code above confirms that two linear layers equal one linear layer. Extend the scaffold to $L = 5$ layers and verify the same property holds.
>
> **(b)** Now suppose $\sigma(z) = 2z + 1$ (a linear activation, not the identity). Show algebraically — and verify in code — that the network is still equivalent to a single affine map. What are $\widetilde{W}$ and $\widetilde{\mathbf{b}}$ in this case?
>
> **(c)** *(Challenge)* Is the class of functions representable by a network with $L$ *nonlinear* layers strictly larger than the class representable by $L-1$ nonlinear layers? What does the **universal approximation theorem** say?

<a id="part3"></a>
# Part 3: Activation Functions — A Tour of Units

## 3.1 Hidden-layer activations

The activation function determines what each hidden unit "detects" and, crucially, how gradients flow backward during training (more on that in [`backpropagation.ipynb`](backpropagation.ipynb)).

| Name | Formula | Range | Key virtue | Caution |
|------|---------|-------|-----------|---------|
| **Sigmoid** | $\dfrac{1}{1+e^{-z}}$ | $(0,1)$ | Smooth; natural probability interpretation | Saturates for large $|z|$; vanishing gradients |
| **Tanh** | $\tanh(z)$ | $(-1,1)$ | Zero-centred; stronger gradients near 0 | Still saturates at extremes |
| **ReLU** | $\max(0,z)$ | $[0,\infty)$ | Fast; sparse activations; no saturation for $z>0$ | "Dying ReLU": unit stuck at 0 if always $z\le 0$ |
| **Leaky ReLU** | $\max(\alpha z,\,z),\;\alpha\ll 1$ | $\mathbb{R}$ | Fixes dying ReLU with small gradient for $z<0$ | Extra hyperparameter $\alpha$ |
| **GeLU** | $z\,\Phi(z)$ | $\mathbb{R}$ | Smooth; state-of-the-art in transformers (BERT, GPT) | Slightly costlier to evaluate |

Here $\Phi(z) = \frac{1}{\sqrt{2\pi}}\int_{-\infty}^{z} e^{-t^2/2}\,dt$ is the standard normal CDF.

**A note on GeLU.** $\text{GeLU}(z) = z \cdot \Phi(z)$ can be read as *soft stochastic gating*: the input $z$ is multiplied by the probability that a standard normal $N \le z$. For $z \gg 0$ this is $\approx z$ (acts like the identity, same as ReLU); for $z \ll 0$ it is $\approx 0$ (same as ReLU). Unlike ReLU, GeLU is smooth everywhere — its derivative exists at $z=0$ — which is believed to help optimisation.

## 3.2 Output-layer units: Softmax

For multi-class classification the output layer uses **softmax**, which turns a vector of *logits* $\mathbf{z}^{(L)} \in \mathbb{R}^K$ into a probability distribution over $K$ classes:

$$\boxed{\hat{y}_k = \mathrm{softmax}(\mathbf{z}^{(L)})_k = \frac{e^{z_k^{(L)}}}{\displaystyle\sum_{j=1}^{K} e^{z_j^{(L)}}}, \qquad k = 1, \ldots, K.}$$

The outputs satisfy $\hat{y}_k \ge 0$ and $\sum_k \hat{y}_k = 1$.
In `mnist_ff.ipynb` the output layer had $K = 10$ (one per digit class); Keras's `SparseCategoricalCrossentropy(from_logits=True)` applies softmax internally, which is why the final `Dense(10)` layer had no explicit activation.

In [ ]:
# Activation function definitions
z = np.linspace(-4.5, 4.5, 500)

def sigmoid(z):    return 1 / (1 + np.exp(-z))
def d_sigmoid(z):  s = sigmoid(z); return s * (1 - s)
def relu(z):       return np.maximum(0, z)
def d_relu(z):     return (z > 0).astype(float)
def lrelu(z, a=0.1): return np.where(z > 0, z, a * z)
def d_lrelu(z, a=0.1): return np.where(z > 0, 1.0, a)
def gelu(z):       return z * ndtr(z)
def d_gelu(z):     return ndtr(z) + z * _norm.pdf(z)

acts = [
    ('Sigmoid',     sigmoid, d_sigmoid, '#3498db'),
    ('Tanh',        np.tanh, lambda z: 1 - np.tanh(z)**2, '#e74c3c'),
    ('ReLU',        relu,    d_relu,    '#2ecc71'),
    ('Leaky ReLU',  lrelu,   d_lrelu,   '#f39c12'),
    ('GeLU',        gelu,    d_gelu,    '#9b59b6'),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, fn, dfn, c in acts:
    axes[0].plot(z, fn(z),  color=c, lw=2.2, label=name)
    axes[1].plot(z, dfn(z), color=c, lw=2.2, label=name)

for ax, title in zip(axes, [r'Activation functions $\sigma(z)$',
                              r"Derivatives $\sigma'(z)$"]):
    ax.axhline(0, color='k', lw=0.6, ls='--')
    ax.axvline(0, color='k', lw=0.6, ls='--')
    ax.set_xlabel(r'$z$', fontsize=13)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10)
    ax.set_xlim(-4.5, 4.5)

axes[0].set_ylim(-1.3, 2.0)
axes[1].set_ylim(-0.15, 1.2)
plt.tight_layout()
plt.show()

**Observations to make:**
- **Sigmoid and Tanh** both saturate: their derivatives $\to 0$ as $|z| \to \infty$. A unit in this regime receives almost no gradient signal — the **vanishing gradient** problem.
- **ReLU** has derivative exactly 1 for $z > 0$ (no saturation), but exactly 0 for $z < 0$ (potential dead units).
- **Leaky ReLU** keeps a small slope $\alpha$ for $z < 0$, preventing dead units.
- **GeLU** is smooth and interpolates between 0 and $z$; its derivative is positive everywhere (no dead units) but less than 1 near $z = 0$ (some saturation-like behaviour).

**Experiment:** The interactive below highlights one activation against the ReLU baseline. Pay attention to:
- Where does the derivative approach zero (vanishing gradient region)?
- Is the function differentiable at $z = 0$?

In [ ]:
def _plot_one_act(activation='GeLU', show_derivative=True):
    fn_map = {'Sigmoid': (sigmoid, d_sigmoid, '#3498db'),
              'Tanh':    (np.tanh, lambda z: 1 - np.tanh(z)**2, '#e74c3c'),
              'ReLU':    (relu,    d_relu,    '#2ecc71'),
              'Leaky ReLU': (lrelu, d_lrelu, '#f39c12'),
              'GeLU':    (gelu,    d_gelu,    '#9b59b6')}
    fn, dfn, color = fn_map[activation]

    fig, ax = plt.subplots(figsize=(7.5, 4.5))
    ax.plot(z, relu(z), color='#cccccc', lw=1.5, ls='--', label='ReLU (reference)')
    ax.plot(z, fn(z), color=color, lw=2.5, label=activation)
    if show_derivative:
        ax.plot(z, dfn(z), color=color, lw=2, ls=':', alpha=0.8,
                label=f"{activation}' (derivative)")
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_xlabel(r'$z$', fontsize=13)
    ax.set_ylim(-1.4, 2.3)
    ax.set_xlim(-4.5, 4.5)
    ax.legend(fontsize=11)
    ax.set_title(f'{activation} vs ReLU (dashed = derivative)', fontsize=13)
    plt.tight_layout(); plt.show()

interact(_plot_one_act,
         activation=Dropdown(options=['Sigmoid','Tanh','ReLU','Leaky ReLU','GeLU'],
                              value='GeLU', description='Activation:'),
         show_derivative=True);

## 3.3 Softmax in action

Softmax converts logits into probabilities. Drag the logit sliders to see how the probability mass shifts.
Notice the **sharpening effect**: increasing one logit relative to the others concentrates nearly all probability mass on that class.

In [ ]:
def _softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()

def _plot_softmax(z1=2.0, z2=0.5, z3=-0.5, z4=-1.5, z5=-2.0):
    logits = np.array([z1, z2, z3, z4, z5])
    probs  = _softmax(logits)
    labels = [f'C{k+1}\n$z={v:.1f}$' for k, v in enumerate(logits)]
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].bar(labels, logits, color=colors, edgecolor='k', lw=0.5)
    axes[0].axhline(0, color='k', lw=0.6)
    axes[0].set_ylabel(r'Logit $z_k$')
    axes[0].set_title(r'Input logits $\mathbf{z}$')

    bars = axes[1].bar(labels, probs, color=colors, edgecolor='k', lw=0.5)
    for bar, p in zip(bars, probs):
        axes[1].text(bar.get_x() + bar.get_width() / 2, p + 0.01,
                     f'{p:.3f}', ha='center', fontsize=9)
    axes[1].set_ylim(0, 1.05)
    axes[1].set_ylabel(r'Probability $\hat{y}_k$')
    axes[1].set_title(r'Softmax output $\hat{\mathbf{y}}$')
    plt.tight_layout(); plt.show()

interact(_plot_softmax,
         z1=FloatSlider(value=2.0,  min=-4, max=4, step=0.1, description=r'$z_1$:'),
         z2=FloatSlider(value=0.5,  min=-4, max=4, step=0.1, description=r'$z_2$:'),
         z3=FloatSlider(value=-0.5, min=-4, max=4, step=0.1, description=r'$z_3$:'),
         z4=FloatSlider(value=-1.5, min=-4, max=4, step=0.1, description=r'$z_4$:'),
         z5=FloatSlider(value=-2.0, min=-4, max=4, step=0.1, description=r'$z_5$:'));

<a id="part4"></a>
# Part 4: Tracing a Forward Pass End-to-End

Let us trace a complete forward pass through a concrete small network: architecture $(3 \to 4 \to 2)$, ReLU hidden units, softmax output.

$$\mathbf{x} \in \mathbb{R}^3
  \;\xrightarrow{W^{(1)},\,\mathbf{b}^{(1)}}\;
  \mathbf{z}^{(1)} \in \mathbb{R}^4
  \;\xrightarrow{\mathrm{ReLU}}\;
  \mathbf{a}^{(1)} \in \mathbb{R}^4
  \;\xrightarrow{W^{(2)},\,\mathbf{b}^{(2)}}\;
  \mathbf{z}^{(2)} \in \mathbb{R}^2
  \;\xrightarrow{\mathrm{softmax}}\;
  \hat{\mathbf{y}} \in \mathbb{R}^2$$

This is the same structure as the MNIST model, just with $n_0 = 3$ (instead of 784) and $n_2 = 2$ (instead of 10).

In [ ]:
np.set_printoptions(precision=4, suppress=True)
rng3 = np.random.default_rng(0)

# Random network parameters (illustrative)
W1 = rng3.standard_normal((4, 3)) * 0.5
b1 = np.zeros(4)
W2 = rng3.standard_normal((2, 4)) * 0.5
b2 = np.zeros(2)

x = np.array([0.5, -1.2, 0.8])   # input vector

sep = '=' * 52
print(sep)
print('FORWARD PASS   network shape  (3 → 4 → 2)')
print(sep)

# --- Layer 1 ---
a0 = x
z1 = W1 @ a0 + b1
a1 = relu(z1)
print(f'\nLayer 1  (ReLU hidden,  n₁ = 4)')
print(f'  z⁽¹⁾ = W⁽¹⁾ x + b⁽¹⁾  =  {z1}')
print(f'  a⁽¹⁾ = ReLU(z⁽¹⁾)    =  {a1}')
print(f'  units zeroed by ReLU:  {(z1 < 0).sum()} of {len(z1)}')

# --- Layer 2 ---
z2 = W2 @ a1 + b2
y_hat = _softmax(z2)
print(f'\nLayer 2  (softmax output,  n₂ = 2)')
print(f'  z⁽²⁾ = W⁽²⁾ a⁽¹⁾ + b⁽²⁾  =  {z2}')
print(f'  ŷ = softmax(z⁽²⁾)      =  {y_hat}')
print(f'  Σ ŷₖ = {y_hat.sum():.6f}   (must equal 1)')

# Exercises

> **Exercise 4.1.** *(Counting parameters)*
>
> **(a)** Derive a formula for the total number of trainable parameters (weights + biases) in a network with architecture $(n_0, n_1, \ldots, n_L)$.
>
> **(b)** Apply your formula to $(784, 64, 64, 10)$ and verify it matches the $55{,}050$ reported in `mnist_ff.ipynb`.
>
> **(c)** If both hidden layers are widened to 128 units — architecture $(784, 128, 128, 10)$ — by what factor does the total parameter count increase?

> **Exercise 4.2.** *(Choosing activations)*
>
> For each task below, state the appropriate output activation and explain why.
>
> **(a)** Classify an image as exactly one of $K = 1{,}000$ categories.
>
> **(b)** Detect which of $K = 10$ attributes are present in an image (multiple attributes can be present simultaneously).
>
> **(c)** Predict a continuous quantity such as house price or temperature.

> **Exercise 4.3.** *(Vanishing gradients — experiment)*
>
> The scaffold below propagates a gradient signal backward through a deep network and records its norm at each layer. Run it with `act = 'sigmoid'` and then with `act = 'relu'`.
>
> **(a)** Describe what happens to the gradient norm as you go deeper with sigmoid. Why?
>
> **(b)** Does the same phenomenon occur with ReLU? Explain the difference using the derivative plots from Part 3.
>
> **(c)** *(Challenge)* Modify the weight initialisation (currently `scale = 0.5`) to mitigate the vanishing gradient for sigmoid. What scale makes the gradient norms roughly constant across layers?

In [ ]:
depth  = 10    # @param {type:"integer"}
width  = 40    # @param {type:"integer"}
act    = 'sigmoid'   # @param ["sigmoid", "relu", "tanh", "gelu"]
scale  = 0.5   # @param {type:"number"}

_act  = {'sigmoid': sigmoid, 'relu': relu,
         'tanh': np.tanh,   'gelu': gelu}[act]
_dact = {'sigmoid': d_sigmoid, 'relu': d_relu,
         'tanh': lambda z: 1 - np.tanh(z)**2,
         'gelu': d_gelu}[act]

rng4 = np.random.default_rng(1)
Ws = [rng4.standard_normal((width, width)) * scale for _ in range(depth)]
x0 = rng4.standard_normal(width)

# Forward pass
zs, as_ = [], [x0]
for W in Ws:
    zs.append(W @ as_[-1])
    as_.append(_act(zs[-1]))

# Backward pass  (loss = sum of outputs; upstream seed = ones)
grad = np.ones(width)
norms = []
for l in reversed(range(depth)):
    grad = grad * _dact(zs[l])   # local gradient through activation
    grad = Ws[l].T @ grad        # local gradient through linear map
    norms.insert(0, np.linalg.norm(grad))

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(1, depth + 1), norms, 'o-', color='#3498db', lw=2, ms=5)
ax.set_xlabel('Layer depth (1 = closest to output)')
ax.set_ylabel('Gradient norm  (log scale)')
ax.set_title(f'Gradient signal through depth — activation: {act},  init scale: {scale}')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Summary

| Concept | Key formula / takeaway |
|---------|------------------------|
| **Unit (pre-activation)** | $\mathbf{z}^{(l)} = W^{(l)}\mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$ |
| **Unit (activation)** | $\mathbf{a}^{(l)} = \sigma(\mathbf{z}^{(l)})$, element-wise |
| **No nonlinearity** | Stacking linear layers $=$ one linear layer |
| **Sigmoid / Tanh** | Smooth, bounded; saturate for large $|z|$ $\Rightarrow$ vanishing gradients |
| **ReLU** | $\max(0,z)$; fast, sparse; risk of dying units |
| **GeLU** | $z\,\Phi(z)$; smooth; standard in transformer architectures |
| **Softmax** | Vector output: $\hat{y}_k = e^{z_k}/\sum_j e^{z_j}$, sums to 1 |

**Next:** [`backpropagation.ipynb`](backpropagation.ipynb) shows how to compute $\partial\mathcal{L}/\partial W^{(l)}$ for every weight in the network — efficiently, in a single backward pass.